In [2]:
!pip install xgboost


In [3]:
from sklearn.ensemble import StackingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier

import pandas as pd
import os
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier



In [4]:
train_path = os.path.join('..','data', 'processed', 'cleaned_train_data_logScaled.csv')
test_path = os.path.join('..', 'data', 'processed', 'cleaned_test_data_logScaled.csv')

trainDF = pd.read_csv(train_path)
testDF = pd.read_csv(test_path)

mainDF = pd.concat([trainDF, testDF], axis=0, ignore_index=True)

In [5]:
X_train = trainDF.drop(columns=['Y', 'X18'], axis=1)
y_train = trainDF['Y']

X_test = testDF.drop(columns=['X18'], axis=1)

In [6]:

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)


In [ ]:
# Base learners
base_learners = [
    ('catboost', CatBoostClassifier(verbose=0, class_weights=[1, 4])),
    ('xgb', XGBClassifier(scale_pos_weight=4, use_label_encoder=False, eval_metric='logloss')),
    ('rf', RandomForestClassifier(class_weight='balanced'))
]

meta_learner = LogisticRegression()

# Stacking ensemble
stacked_model = StackingClassifier(
    estimators=base_learners,
    final_estimator=meta_learner,
    cv=5
)

# Train
stacked_model.fit(X_train, y_train)


c:\Users\raadr\anaconda3\Lib\site-packages\xgboost\training.py:183: UserWarning: [06:36:13] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
c:\Users\raadr\anaconda3\Lib\site-packages\xgboost\training.py:183: UserWarning: [06:43:16] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
c:\Users\raadr\anaconda3\Lib\site-packages\xgboost\training.py:183: UserWarning: [06:43:17] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
c:\Users\raadr\anaconda3\Lib\site-packages\xgboost\training.py:183: UserWarning: [06:43:18] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtr

In [ ]:
y_pred = stacked_model.predict(X_test)

In [ ]:
output = pd.DataFrame({'ID': testDF['ID'], 'Prediction': y_pred})


output.to_csv('predictions.csv', index=False)